In [ ]:
import torch
from torch import nn
import triton
import triton.language as tl
import gc

import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s"
)


In [ ]:
@triton.jit
def _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v,
                             offset_y, dtype: tl.dtype, start_m, qk_scale,
                             block_m: tl.constexpr, hidden_dim: tl.constexpr, block_n: tl.constexpr, stage: tl.constexpr,
                             offs_m: tl.constexpr, offs_n: tl.constexpr, n_ctx: tl.constexpr, non_mask: tl.constexpr, warp_specialize: tl.constexpr):
    # print(f"Stage : {stage}")
    if stage == 1:
        lo, hi = 0, start_m*block_m
    elif stage == 2:
        lo, hi = start_m*block_m, (start_m+1)*block_m
    else:
        lo, hi = 0, n_ctx

    if non_mask:
        lo, hi = 0, (start_m+1)*block_m

    offsetk_y = offset_y + lo
    offsetv_y = offset_y + lo
    for start_n in tl.range(lo, hi, block_n, warp_specialize=warp_specialize):
        # print(f"start_n : {start_n} : offset of k [{offsetk_y},0]")
        k = tl.trans(desc_k.load([offsetk_y, 0]))
        qk = tl.dot(q, k) * qk_scale
        if stage == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            qk = qk  + tl.where(mask, 0, -1.0e6)
        m_ij = tl.maximum(m_i, tl.max(qk, 1))
        qk -= m_ij[:, None]
        p = tl.math.exp(qk)
        # -- compute correction factor
        alpha = tl.math.exp(m_i - m_ij)
        l_ij = tl.sum(p, 1)
        acc = acc * alpha[:, None]

        # print(f"Offset of v [0, {offsetv_y}]")
        v = desc_v.load([offsetv_y, 0])
        p = p.to(dtype)
        acc += tl.dot(p, v)
        l_i = l_i * alpha + l_ij
        m_i = m_ij
        offsetk_y += block_n
        offsetv_y += block_n
    return acc, l_i, m_i

@triton.autotune(
    configs=[
        triton.Config({'block_m':32, 'block_n':16}, num_warps=4, num_stages=1),
        triton.Config({'block_m':16, 'block_n':32}, num_warps=4, num_stages=1),
        triton.Config({'block_m':32, 'block_n':32}, num_warps=4, num_stages=1),
        triton.Config({'block_m':16, 'block_n':16}, num_warps=4, num_stages=1)
    ],
    key=['n_ctx', 'hidden_dim'],   # runtime-dependent shapes
)
@triton.jit
def _attention_forward(sm_scale, max_tensor, softmax_dem, batch, num_heads, n_ctx, desc_q, desc_k, desc_v, desc_o, lower_precision: tl.constexpr,
                       hidden_dim: tl.constexpr, mask_region: tl.constexpr, warp_specialize: tl.constexpr,
                       block_m: tl.constexpr, block_n: tl.constexpr):
    dtype: tl.dtype = tl.float16 if lower_precision else tl.float32
    assert block_n <= hidden_dim
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    batch_idx = off_hz // num_heads
    head_idx = off_hz % num_heads
    # print(f"start_m : {start_m}, off_h : {off_h}")
    y_dim = num_heads * n_ctx * batch
    desc_q = tl.make_tensor_descriptor(desc_q, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_m, hidden_dim])
    desc_v = tl.make_tensor_descriptor(desc_v, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                         block_shape=[block_n, hidden_dim])
    desc_k = tl.make_tensor_descriptor(desc_k, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_n, hidden_dim])
    desc_o = tl.make_tensor_descriptor(desc_o, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_m, hidden_dim])
    offset_y = batch_idx*num_heads*n_ctx + head_idx*n_ctx
    # print(f"offset_y : {offset_y}")
    qo_offset_y = offset_y + start_m*block_m
    # print(f"qo_offset_y : {qo_offset_y}")
    offs_m = start_m*block_m + tl.arange(0, block_m)
    offs_n = tl.arange(0, block_n)
    # print(f"offs_m : {offs_m}")
    # print(f"offs_n : {offs_n}")
    # initialize pointer to m and l
    m_i = tl.zeros([block_m], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([block_m], dtype=tl.float32) + 1.0
    acc = tl.zeros([block_m, hidden_dim], dtype=tl.float32)
    # load scales
    qk_scale = sm_scale
    # qk_scale *= 1.44269504 #1/log(2)
    q = desc_q.load([qo_offset_y,0])
    # print(f"q load : {[qo_offset_y, 0]}")
    if mask_region:
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 1, offs_m, offs_n, n_ctx, False, warp_specialize)
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 2, offs_m, offs_n, n_ctx, False, warp_specialize)
    else:
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 2, offs_m, offs_n, n_ctx, True, warp_specialize)
    m_i += tl.math.log(l_i)
    acc = acc / l_i[:, None]
    off_hz = batch_idx*num_heads*n_ctx + head_idx*n_ctx
    m_ptrs = max_tensor + off_hz + offs_m
    sft_dem_ptrs = softmax_dem + off_hz + offs_m
    tl.store(m_ptrs, m_i)
    tl.store(sft_dem_ptrs, l_i)
    desc_o.store([qo_offset_y, 0], acc.to(dtype))

In [ ]:
@triton.jit
def _attention_bwd_pre_process(o_ptr, do_ptr, delta_ptr,
                               batch: tl.constexpr,
                               n_ctx: tl.constexpr,
                               pre_block: tl.constexpr,
                               heads: tl.constexpr,
                               hidden: tl.constexpr):
    pre_block_per_nctx = tl.program_id(0)
    off_hz = tl.program_id(1)
    head_idx = off_hz % heads
    batch_idx = off_hz // heads
    offs_pre_block = pre_block_per_nctx*pre_block + tl.arange(0, pre_block)
    offs_hid = tl.arange(0, hidden)
    offset = batch_idx*heads*n_ctx*hidden + head_idx*n_ctx*hidden + offs_pre_block[:,None]*hidden + offs_hid[None,:]
    o = tl.load(o_ptr + offset)
    do = tl.load(do_ptr + offset)
    o = o.to(tl.float32)
    do = do.to(tl.float32)
    o = tl.maximum(tl.minimum(o, 1.0e8), -1.0e8)
    do = tl.maximum(tl.minimum(do, 1.0e8), -1.0e8)
    o_do = tl.sum(o*do, axis=1)
    delta = delta_ptr + batch_idx*heads*n_ctx + head_idx*n_ctx + offs_pre_block
    tl.store(delta, o_do)

@triton.jit
def _attention_bwd_dkdv(dkey, dvalue, m, d, q, k, v, do,
                        init_offset, offset_along_n, offset_along_h, block_m, block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=False, mask=False):
    base_mask = head_idx*n_ctx + batch_idx*num_heads*n_ctx
    mask_offset_along_m = ctxid*block_m + base_mask + tl.arange(0, block_m)
    mask_offset_along_n = ctxid*block_m + base_mask + tl.arange(0, block_n)
    if causal and mask:
      num_steps = block_m // block_n
    elif causal and not mask:
      num_steps = (n_ctx - ctxid*block_m) // block_n
      num_steps = num_steps - block_m 
      mask_offset_along_n = ctxid*block_m + block_m + base_mask + tl.arange(0, block_n)
    else:
      num_steps = n_ctx // block_n
    
    offset_block_n = init_offset + offset_along_n[:, None] + offset_along_h[None, :]
    offset_block_n_T = init_offset + offset_along_n[None, :] + offset_along_h[:, None]
    for _ in range(num_steps):
      queryT = tl.load(q + offset_block_n_T)  # pre-load to L1
      d_of_o = tl.load(do + offset_block_n)  # pre-load to L1
      max_tensor = tl.load(m + mask_offset_along_n)  # pre-load to L1
      kqT = tl.dot(k, queryT)*sm_scale
      pT = tl.exp(kqT - max_tensor[None,:])
      if mask:
        mask_tensor = (mask_offset_along_m[:, None] <= mask_offset_along_n[None, :])
        pT += tl.where(mask_tensor, 0, 0.0)

      dvalue += tl.dot(pT, d_of_o).to(tl.float32)
      dpT = tl.dot(v, tl.trans(d_of_o)).to(tl.float32)
      delta = tl.load(d + mask_offset_along_n)  # pre-load to L1
      dsT = pT * (dpT - delta[None,:])
      dkey += tl.dot(dsT, tl.trans(queryT))

      mask_offset_along_n += block_n
      offset_block_n += block_n*hidden_dim
      offset_block_n_T += block_n*hidden_dim
    return dkey, dvalue

@triton.jit
def _attention_bwd_dq(dquery, m, d, q, k, v, do,
                        offset_batch_head, offset_along_n, offset_along_h, block_m, block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=False, mask=False):
    if causal and not mask and ctxid == 0:
      return dquery
    base_mask = head_idx*n_ctx + batch_idx*num_heads*n_ctx
    mask_offset_along_m = ctxid*block_m + base_mask + tl.arange(0, block_m)
    if causal and mask:
      num_steps = block_m // block_n
      mask_offset_along_n = ctxid*block_m + base_mask + tl.arange(0, block_n)
      offset_block_n_T = ctxid*block_m*hidden_dim + offset_batch_head + offset_along_n[None, :] + offset_along_h[:, None]
    elif causal and not mask:
      num_steps = ctxid*block_m // block_n
      offset_block_n_T = offset_batch_head + offset_along_n[None, :] + offset_along_h[:, None]
    else:
      num_steps = n_ctx // block_n
    
    max_tensor = tl.load(m + mask_offset_along_m)  # pre-load to L1
    delta = tl.load(d + mask_offset_along_m)  # pre-load to L1
    
    for _ in range(num_steps):
      keyT = tl.load(k + offset_block_n_T)  # pre-load to L1
      valueT = tl.load(v + offset_block_n_T)  # pre-load to L1
      qkT = tl.dot(q, keyT)*sm_scale
      p = tl.exp(qkT - max_tensor[:, None])
      if mask:
        mask_tensor = (mask_offset_along_m[:, None] >= mask_offset_along_n[None, :])
        p += tl.where(mask_tensor, 0, 0.0)
        # increment only in case of mask
        mask_offset_along_n += block_n
      dp = tl.dot(do, valueT).to(tl.float32)
      ds = p * (dp - delta[:, None])
      dquery += tl.dot(ds, tl.trans(keyT))
      offset_block_n_T += block_n*hidden_dim
    return dquery

@triton.autotune(
    configs=[
        triton.Config({'block_m':32, 'block_n':32}, num_warps=4, num_stages=1),
        triton.Config({'block_m':16, 'block_n':16}, num_warps=4, num_stages=1),
        triton.Config({'block_m':64, 'block_n':32}, num_warps=4, num_stages=1),
        triton.Config({'block_m':64, 'block_n':16}, num_warps=4, num_stages=1)
    ],
    key=['n_ctx', 'hidden_dim'],   # runtime-dependent shapes
)
@triton.jit
def _attention_bwd(q, k, v, do, dq, dk, dv, m, d,
                   sm_scale: tl.constexpr, num_heads: tl.constexpr,
                   n_ctx: tl.constexpr, hidden_dim: tl.constexpr, bulk_slice_factor: tl.constexpr, block_m: tl.constexpr,
                   block_n: tl.constexpr, CAUSAL: tl.constexpr = True):
    # LN2 = 0.6931471824645996  # = ln(2)
    # current context block
    ctxid = tl.program_id(0)
    # current head
    hzid = tl.program_id(1)
    batch_idx = hzid // num_heads
    head_idx = hzid % num_heads
    # init offset can be used for both dkdv & dq
    offset_batch_head = head_idx*n_ctx*hidden_dim + batch_idx*num_heads*n_ctx*hidden_dim
    init_offset = ctxid*block_m*hidden_dim + offset_batch_head

    dvalue = tl.zeros([block_m, hidden_dim], dtype=tl.float32)
    dkey = tl.zeros([block_m, hidden_dim], dtype=tl.float32)

    offset_along_m = tl.arange(0, block_m)*hidden_dim
    offset_along_n = tl.arange(0, block_n)*hidden_dim
    offset_along_h = tl.arange(0, hidden_dim)
    
    offset_block_m = init_offset + offset_along_m[:, None] + offset_along_h[None, :]
    key = tl.load(k + offset_block_m)
    value = tl.load(v + offset_block_m)
    assert block_m % block_n == 0, "block_m should be divisible by block_n"

    if CAUSAL:
      # for the mask part of block_m
      dkey, dvalue = _attention_bwd_dkdv(dkey, dvalue, m, d, q, key, value, do, init_offset,
                          offset_along_n, offset_along_h, block_m, block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=True, mask=True)
      # for the non-mask part for data before the mask (past data)
      dkey, dvalue = _attention_bwd_dkdv(dkey, dvalue, m, d, q, key, value, do, init_offset,
                          offset_along_n, offset_along_h, block_m, block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=True, mask=False)
    else:
      # for non-causal, we feed both the past and future data together as there is no mask
      dkey, dvalue = _attention_bwd_dkdv(dkey, dvalue, m, d, q, key, value, do, init_offset,
                          offset_along_n, offset_along_h, block_m,block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=False, mask=False)
    
    tl.store(dv + offset_block_m, dvalue)
    tl.store(dk + offset_block_m, dkey*sm_scale)

    dquery = tl.zeros([block_m, hidden_dim], dtype=tl.float32)
    dervative_o = tl.load(do + offset_block_m)
    query = tl.load(q + offset_block_m)
    if CAUSAL:
      # for the mask part of block_m
      dquery = _attention_bwd_dq(dquery, m, d, query, k, v, dervative_o, offset_batch_head,
                          offset_along_n, offset_along_h, block_m, block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=True, mask=True)
      # for the non-mask part for data before the mask (past data)
      dquery = _attention_bwd_dq(dquery, m, d, query, k, v, dervative_o, offset_batch_head,
                          offset_along_n, offset_along_h, block_m, block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=True, mask=False)
    else:
      # for non-causal, we feed both the past and future data together as there is no mask
      dquery = _attention_bwd_dq(dquery, m, d, query, k, v, dervative_o, offset_batch_head,
                          offset_along_n, offset_along_h, block_m,block_n, batch_idx, head_idx, ctxid, num_heads, n_ctx, hidden_dim, sm_scale, causal=False, mask=False)
    tl.store(dq + offset_block_m, dquery*sm_scale)
    

In [ ]:
from torch._prims_common import Dim
class _attention(torch.autograd.Function):

  # Assumption that it is used only for causal case
  @staticmethod
  def forward(ctx, q, k, v, world_size=1, rank=0, warp_specialize=True):
        batch, num_heads, n_ctx, hidden_dim = q.shape[0], q.shape[1], q.shape[2], q.shape[3]
        sm_scale = 1.0 / (hidden_dim ** 0.5)
        o = torch.zeros_like(q)
        M = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
        sft_d = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
        grid_fwd = lambda META: (
            triton.cdiv(n_ctx, META['block_m']),
            num_heads * batch,
            1
        )
        if q.dtype == torch.float16:
          lower_precision = True
        else:
          lower_precision = False
        # grid = (n_ctx//block_m, num_heads*batch, 1)
        # print(f"Grid : {grid}")

        # Attention forward
        # mask region
        _attention_forward[grid_fwd](sm_scale, M, sft_d, batch, num_heads, n_ctx,
                        q, k, v, o, lower_precision,
                        hidden_dim, True, warp_specialize,)
        ctx.save_for_backward(q,k,v,o,M,sft_d)
        ctx.sm_scale = sm_scale
        ctx.hidden_dim = hidden_dim
        ctx.rank = rank
        ctx.world_size = world_size
        ctx.num_heads = num_heads
        ctx.batch = batch
        # print(o)
        return o

  @staticmethod
  def backward(ctx, do):
      q, k, v, o, M, sft_d = ctx.saved_tensors
      sm_scale = ctx.sm_scale
      hidden_dim = ctx.hidden_dim
      rank = ctx.rank
      world_size = ctx.world_size
      num_heads = ctx.num_heads
      batch = ctx.batch
      pre_block = 64
      num_hiddens = q.shape[-1]
      n_ctx = q.shape[2]
      print(f"[SFT_DENOMINATOR] {sft_d.shape}")
      # print(f"q : {q.shape} : N_CTX : {n_ctx}, PRE_BLOCK : {pre_block}")
      assert n_ctx >= pre_block and n_ctx % pre_block == 0
      grid_preprocess = (n_ctx//pre_block, num_heads*batch, 1)
      # print(f"Grid (bwd_pre_process) : {grid_preprocess}, q: {q.shape}, k: {k.shape}, v: {v.shape} ")
      delta = torch.zeros_like(M, device=q.device, dtype=torch.float32)
      # logging.debug(f"[Attention] do({do.shape}): {torch.isnan(do).any()} : do.max(): {do.abs().max()} : do.min(): {do.abs().min()}")
      # print(f'[torch matmul] delta: {torch.sum(o*do,dim=-1)}')
      # Preprocess
      _attention_bwd_pre_process[grid_preprocess](o, do, delta, batch, n_ctx, pre_block, num_heads, num_hiddens)
      # logging.debug(f"[Attention] delta({delta.shape}): {torch.isnan(delta).any()} : delta.max(): {delta.abs().max()} : delta.min(): {delta.abs().min()}")
      # print(f"[Attention] delta({delta.shape}): {delta}")
      # bwd
      dq = torch.zeros_like(q)
      dk = torch.zeros_like(k)
      dv = torch.zeros_like(v)
      bulk_slice_factor = 1
      # grid_bwd = (n_ctx//block_m, num_heads*batch, 1)
      # print(f"Grid (bwd) : {grid_bwd}")
      # ctx = torch.arange(0, n_ctx, device=q.device)
      # mask = (ctx[None, :] >= ctx[:, None])
      # scores = torch.matmul(q, k.transpose(-2, -1)) * sm_scale
      # scores = scores.masked_fill(~mask, -1e9)
      # safer: recompute for verification
      # scores = scores - scores.max(dim=-1, keepdim=True).values
      # p = torch.softmax(scores, dim=-1)
      # dv_torch = torch.matmul(p.transpose(-2, -1), do)
      # print(f"[torch matmul] dv: {dv_torch}")
      # print(f"[torch matmul] dv: Minumum gradient :{dv_torch.min()}, Maximum gradient : {dv_torch.max()}")
      gc.collect()
      torch.cuda.empty_cache()
      grid_bwd = lambda META: (
          triton.cdiv(n_ctx, META['block_m']),
          num_heads * batch,
          1
      )
      _attention_bwd[grid_bwd](q, k, v, do, dq, dk, dv, M, delta, sft_d, sm_scale, batch, num_heads, n_ctx,
                               num_hiddens,bulk_slice_factor,)
      # print(f"[Attention] dv {dv}")
      # print(f"[Attention] dv({torch.isnan(dv).any()})")
      # print(f"[Attention] dk({torch.isnan(dk).any()})")
      # print(f"[Attention] dq({torch.isnan(dq).any()})")
      return dq, dk, dv, None, None, None, None, None, None, None, None, None

In [ ]:
class AttentionTorch(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.LazyLinear(TOTAL_VOCAB, bias=False, device=DEVICE)
        self.loss_fn = nn.CrossEntropyLoss(reduction="mean")
        self.attention = torch.nn.functional.scaled_dot_product_attention

    def forward(self, q, k, v, output_token, is_causal=True):
        output = self.attention(q, k, v, is_causal=is_causal)
        output = output.permute(0, 2, 1, 3).reshape(BATCH, N_CTX,-1)
        logits = self.dense(output)
        logits = logits.float()
        # shift BEFORE flatten
        shift_logits = logits[:, :-1, :]
        shift_labels = output_token[:, 1:]

        B, T, H = logits.shape
        logits = logits.view(B*T, H)
        output_token = output_token.view(B*T)
        shift_labels = output_token[...,1:].contiguous()
        shift_logits = logits[...,:-1,:].contiguous()
        loss = self.loss_fn(shift_logits, shift_labels.long())
        return output, loss

class AttentionTriton(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.LazyLinear(TOTAL_VOCAB, bias=False, device=DEVICE)
        self.loss_fn = nn.CrossEntropyLoss(reduction="mean")
        self.attention = _attention.apply

    def forward(self, q, k, v, output_token, is_causal=True):
        output = self.attention(q, k, v)
        output = output.permute(0, 2, 1, 3).reshape(BATCH, N_CTX,-1)
        logits = self.dense(output)
        logits = logits.float()
        # shift BEFORE flatten
        shift_logits = logits[:, :-1, :]
        shift_labels = output_token[:, 1:]

        B, T, H = logits.shape
        logits = logits.view(B*T, H)
        output_token = output_token.view(B*T)
        shift_labels = output_token[...,1:].contiguous()
        shift_logits = logits[...,:-1,:].contiguous()
        loss = self.loss_fn(shift_logits, shift_labels.long())
        return output, loss

# torch.set_printoptions(profile="full")
DEVICE = "cuda"
HIDDENS = 16
N_CTX = 64
TOTAL_VOCAB = 22000
BATCH = 1
attention_module_torch = AttentionTorch()

q_torch = torch.randn([1,1,N_CTX,HIDDENS], device=DEVICE, dtype=torch.float32, requires_grad=True)
k_torch = torch.randn([1,1,N_CTX,HIDDENS], device=DEVICE, dtype=torch.float32, requires_grad=True)
v_torch = torch.randn([1,1,N_CTX,HIDDENS], device=DEVICE, dtype=torch.float32, requires_grad=True)
output_token_torch = torch.randint(0, TOTAL_VOCAB, [1,1,N_CTX], device=DEVICE)
q_triton = q_torch.clone().detach().requires_grad_(True)
k_triton = k_torch.clone().detach().requires_grad_(True)
v_triton = v_torch.clone().detach().requires_grad_(True)
output_token_triton = output_token_torch.clone().detach()
attention_module_triton = AttentionTriton()

output_torch, loss_torch = attention_module_torch(q_torch, k_torch, v_torch, output_token_torch)
print(loss_torch)
loss_torch.backward()
print(f'dq : Minumum gradient : {q_torch.grad.dtype} : {q_torch.grad.min()}, Maximum gradient : {q_torch.max()}')
print(f'dk : Minumum gradient : {k_torch.grad.dtype} :{k_torch.grad.min()}, Maximum gradient : {k_torch.max()}')
print(f'dv : Minumum gradient : {v_torch.grad.dtype} :{v_torch.grad.min()}, Maximum gradient : {v_torch.max()}')

output_triton, loss_triton = attention_module_triton(q_triton, k_triton, v_triton, output_token_triton)
print(loss_triton)
# print(f'o torch : {output_torch}')
# print(f'o triton : {output_triton}')
loss_triton.backward()
print(f'dq : Minumum gradient : {q_triton.grad.dtype} : {q_triton.grad.min()}, Maximum gradient : {q_triton.max()}')
print(f'dk : Minumum gradient : {k_triton.grad.dtype} : {k_triton.grad.min()}, Maximum gradient : {k_triton.max()}')
print(f'dv : Minumum gradient : {v_triton.grad.dtype} : {v_triton.grad.min()}, Maximum gradient : {v_triton.max()}')
# torch.set_printoptions(profile="full")
# print(v_triton.grad)
# torch.set_printoptions(profile="default")
# print(torch.allclose(q_torch.grad, q_triton.grad , atol=1e-5))
# print(torch.allclose(k_torch.grad, k_triton.grad, atol=1e-5))
# print(torch.allclose(v_torch.grad, v_triton.grad, atol=1e-5))

tensor(10.0504, device='cuda:0', grad_fn=<NllLossBackward0>)
dq : Minumum gradient : torch.float32 : -0.002362047089263797, Maximum gradient : 3.5606000423431396
dk : Minumum gradient : torch.float32 :-0.0036443443968892097, Maximum gradient : 3.5358612537384033
dv : Minumum gradient : torch.float32 :-0.0055923485197126865, Maximum gradient : 3.5358612537384033
tensor(10.0489, device='cuda:0', grad_fn=<NllLossBackward0>)
q : torch.Size([1, 1, 64, 16]) : N_CTX : 64, PRE_BLOCK : 64
[Attention] dv(True)
[Attention] dk(False)
[Attention] dq(False)
dq : Minumum gradient : torch.float32 : -2.8455541133880615, Maximum gradient : 3.5606000423431396
dk : Minumum gradient : torch.float32 : -3.5615763664245605, Maximum gradient : 3.5358612537384033
dv : Minumum gradient : torch.float32 : nan, Maximum gradient : 2.698018789291382


In [ ]:
# inp1 = torch.randn([1,1,64,64])
# inp2 = torch.randn([1,1,64,16])
# output = torch.matmul(inp1, inp2)
# print(output.shape)

# ctx = torch.arange(0,64)
# mask = (ctx[None, :] <= ctx[:, None])
# print(mask)
import torch
a = torch.arange(0,64)
print(a[None, :])

In [ ]:
import torch

block_m = 32
block_n = 16
hidden_dim = 32
n_ctx = 64
# ctxid = 0
# head_idx = 0
# batch_idx = 0
batch = 2
num_heads = 4
for batch_idx in range(2):
  for head_idx in range(4):
    for ctxid in range(n_ctx//block_m):
      print(f"------ctxid-{ctxid}---head_idx-{head_idx}-----batch_idx-{batch_idx}---------")
      init_offset = ctxid*block_m*hidden_dim + head_idx*n_ctx*hidden_dim + batch_idx*num_heads*n_ctx*hidden_dim
      y_dim = batch * num_heads * n_ctx * hidden_dim
      offset_m = init_offset + torch.arange(0, block_m)

      row_offset_block_m = torch.arange(0, block_m)*hidden_dim
      col_offset_block_m = torch.arange(0, hidden_dim)
      offset_block_m = init_offset + row_offset_block_m[:, None] + col_offset_block_m[None, :]
      torch.set_printoptions(profile="full")
      # print(offset_block_m)
      # left of mask for non mask regions
      start = 0
      end = ctxid*block_m
      increment = block_n
      # print(f"start {start}, end {end}, increment {increment}")
      row_offset_block_n = torch.arange(0,hidden_dim)*n_ctx
      col_offset_block_n = torch.arange(0, block_n)
      row_offset_block_n_inverted = torch.arange(0, block_n)*hidden_dim
      col_offset_block_n_inverted = torch.arange(0, hidden_dim)
      # query = tl.load(q + offset_block_m)  # pre-load to L1
      # value = tl.load(v + offset_block_m)  # pre-load to L1
      MASK = False
      offset = ctxid*block_m + head_idx*n_ctx*hidden_dim + batch_idx*num_heads*n_ctx*hidden_dim
      offset_inverted = ctxid*block_m*hidden_dim + head_idx*n_ctx*hidden_dim + batch_idx*num_heads*n_ctx*hidden_dim
      for sub_block_n in range(start, end, increment):
        # print(f"-------sub_block_n-----{sub_block_n}-----------------")
        offset += sub_block_n
        offset_n = offset + torch.arange(0, block_n)
        offset_block_n = offset + row_offset_block_n[:, None] + col_offset_block_n[None, :]
        # print(f'Shape : {offset_block_n.shape} : {offset_block_n}')
        offset_inverted += sub_block_n*hidden_dim
        offset_block_n_inverted = offset_inverted + row_offset_block_n_inverted[:, None] + col_offset_block_n_inverted[None, :]
        # print(f'Shape : {offset_block_n_inverted.shape} : {offset_block_n_inverted}')

      # mask regions
      # mask_block_n:tl.constexpr = block_n // bulk_slice_factor
      mask_block_n = block_n
      start = 0
      end = block_m
      increment = mask_block_n
      print(f"start {start}, end {end}, increment {increment}")
      row_offset_block_n = torch.arange(0,hidden_dim)*n_ctx
      col_offset_block_n = torch.arange(0, mask_block_n)
      row_offset_block_n_inverted = torch.arange(0, mask_block_n)*hidden_dim
      col_offset_block_n_inverted = torch.arange(0, hidden_dim)
      MASK = True
      offset = ctxid*block_m + head_idx*n_ctx*hidden_dim + batch_idx*num_heads*n_ctx*hidden_dim
      offset_inverted = ctxid*block_m*hidden_dim + head_idx*n_ctx*hidden_dim + batch_idx*num_heads*n_ctx*hidden_dim
      for sub_block_n in range(start, end, increment):
        print(f"-------sub_block_n-----{sub_block_n}-----------------")
        offset += sub_block_n
        offset_n = offset + torch.arange(0, mask_block_n)
        offset_block_n = offset + row_offset_block_n[:, None] + col_offset_block_n[None, :]
        # print(f'Shape : {offset_block_n.shape} : {offset_block_n}')
        offset_inverted += sub_block_n*hidden_dim
        offset_block_n_inverted = offset_inverted + row_offset_block_n_inverted[:, None] + col_offset_block_n_inverted[None, :]
        # print(f'Shape : {offset_block_n_inverted.shape} : {offset_block_n_inverted}')
        if MASK:
          print((offset_m[:, None] >= offset_n[None, :]))